## **Setting the Environment**

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Setting toolkit folder as working directory

%cd /content/drive/My Drive/Colab Notebooks/Transformer
! ls

In [ ]:
# Importing essential libraries and functions
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, DistilBertForSequenceClassification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from tqdm import tqdm
import pandas as pd


# To plot
import matplotlib.pyplot as plt
%matplotlib inline
import matplotlib as mpl
import seaborn as sns

## **Loading DataSet**

In [ ]:
df = pd.read_json('/content/drive/MyDrive/FYP Approch Transformers/Transformer/AmazonCellReviews.json', lines=True)
df.head(5)

In [ ]:
df.shape #to see the number of columns and rows

In [ ]:
df['overall'].value_counts()

In [ ]:
plt.figure(figsize=(10,8))
ax = sns.countplot(x='overall', data=df)

In [ ]:
def calc_sentiment_with_neutral(overall):
    '''encoding the sentiments of the ratings.'''
    '''This function encodes the rating 1 and 2 as 0, others as 1'''
    if overall >= 3:
        return 1
    else:
        return 0

In [ ]:
df['sentiment'] = df['overall'].apply(calc_sentiment_with_neutral) #applyind function

In [ ]:
df['sentiment'].value_counts() #number of new sentiments

In [ ]:
# Assuming df is your DataFrame containing the sentiment and overall columns along with other columns

# List of columns to keep (sentiment and overall)
columns_to_keep = ["sentiment", "overall", "reviewText"]

# Drop columns other than sentiment and overall
df = df.drop(df.columns.difference(columns_to_keep), axis=1)
df.dropna(subset=['reviewText'], inplace=True) #droping null's in reviews
df.info(verbose=True) #to see the type of columns

In [ ]:
# Checking for missing values

df.isnull().values.any()

In [ ]:
# Let's observe distribution of positive / negative sentiments in dataset

import seaborn as sns
sns.countplot(x='sentiment', data=df)

**UnderSampling**

In [ ]:
from sklearn.utils import resample

# Separate majority and minority classes
df_majority = df[df.sentiment==1]
df_minority = df[df.sentiment==0]

# Downsample majority class
df_majority_downsampled = resample(df_majority,
                                 replace=False,    # sample without replacement
                                 n_samples=138714,     # to match minority class
                                 random_state=123) # reproducible results

# Combine minority class with downsampled majority class
df_downsampled = pd.concat([df_majority_downsampled, df_minority])

# Display new class counts
df_downsampled.sentiment.value_counts()

In [ ]:
# Convert the DataFrame to a JSON string
json_data = df_downsampled.to_json(orient='records', lines=True)

# Save the JSON string to a file
with open('balanced_reviews.json', 'w') as f:
  f.write(json_data)

In [ ]:
df = pd.read_json('balanced_reviews.json', lines=True)
df.head(5)

In [ ]:
# Let's observe distribution of positive / negative sentiments in dataset

import seaborn as sns
sns.countplot(x='sentiment', data=df)

## **Transformer**

In [ ]:

# Load and preprocess the dataset
class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        text = self.texts[index]
        label = self.labels[index]
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(label, dtype=torch.long)
        }

In [ ]:
# Load dataset from CSV (assuming CSV format with 'text' and 'label' columns)
#df = pd.read_csv('path_to_your_dataset.csv')  # replace with your dataset path
texts = df['reviewText'].tolist()
labels = df['sentiment'].tolist()

In [ ]:

# Split data into training and validation sets
train_texts, val_texts, train_labels, val_labels = train_test_split(texts, labels, test_size=0.1)

In [ ]:
# Model hyperparameters
MAX_LEN = 128
BATCH_SIZE = 128
EPOCHS = 2
LEARNING_RATE = 2e-5

In [ ]:
# Initialize tokenizer and model
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)


In [ ]:
# Prepare datasets
train_dataset = SentimentDataset(train_texts, train_labels, tokenizer, MAX_LEN)
val_dataset = SentimentDataset(val_texts, val_labels, tokenizer, MAX_LEN)


In [ ]:
# Prepare DataLoader
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

In [ ]:
# Set device (GPU or CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

In [ ]:
# Optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

**Training**

In [ ]:

# Training loop
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    correct_train_predictions = 0
    total_train_predictions = 0

    for batch in tqdm(train_loader):
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        # Forward pass
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        train_loss += loss.item()

        # Backward pass and optimization
        loss.backward()
        optimizer.step()

        # Predictions and accuracy calculation
        logits = outputs.logits
        predictions = torch.argmax(logits, dim=1)
        correct_train_predictions += torch.sum(predictions == labels)
        total_train_predictions += labels.size(0)

    avg_train_loss = train_loss / len(train_loader)
    train_accuracy = correct_train_predictions.double() / total_train_predictions
    print(f'Epoch {epoch + 1}/{EPOCHS} | Train Loss: {avg_train_loss:.4f} | Train Accuracy: {train_accuracy:.4f}')


**Validation**

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np
for epoch in range(EPOCHS):
  # Validation
  model.eval()
  val_loss = 0
  correct_val_predictions = 0
  total_val_predictions = 0

  all_predictions = []
  all_labels = []

  with torch.no_grad():
     for batch in tqdm(val_loader):
         input_ids = batch['input_ids'].to(device)
         attention_mask = batch['attention_mask'].to(device)
         labels = batch['label'].to(device)

        # Forward pass
         outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
         loss = outputs.loss
         val_loss += loss.item()

        # Predictions and accuracy calculation
         logits = outputs.logits
         predictions = torch.argmax(logits, dim=1)

         correct_val_predictions += torch.sum(predictions == labels)
         total_val_predictions += labels.size(0)

        # Append the predictions and labels for Confusion Matrix and Classification Report
         all_predictions.extend(predictions.cpu().numpy())
         all_labels.extend(labels.cpu().numpy())

avg_val_loss = val_loss / len(val_loader)
val_accuracy = correct_val_predictions.double() / total_val_predictions
print(f'Epoch {epoch + 1}/{EPOCHS} | Validation Loss: {avg_val_loss:.4f} | Validation Accuracy: {val_accuracy:.4f}')

# Generate Confusion Matrix and Classification Report
conf_matrix = confusion_matrix(all_labels, all_predictions)
class_report = classification_report(all_labels, all_predictions)

print("\nConfusion Matrix:\n", conf_matrix)
print("\nClassification Report:\n", class_report)



In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt


# Create confusion matrix
conf_matrix = confusion_matrix(all_labels, all_predictions)


# Create a heatmap with labels
ax = sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues')
ax.set_title('Confusion Matrix')
ax.set_xlabel('Predicted Label')
ax.set_ylabel('True Label')
ax.set_xticklabels(['Negative', 'Positive'], rotation=45)
ax.set_yticklabels(['Negative', 'Positive'])
plt.show()

# Testing our model

In [ ]:

# Then, you can load the model from your drive (replace the path with your own)
Mymodel = DistilBertForSequenceClassification.from_pretrained('/content/drive/MyDrive/FYP Approch Transformers/Transformer/fine_tuned_distilbert_sentiment')
Mytokenizer = AutoTokenizer.from_pretrained('/content/drive/MyDrive/FYP Approch Transformers/Transformer/fine_tuned_distilbert_sentiment')



In [ ]:
# Move Mymodel to the GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
Mymodel.to(device)
Mymodel.eval()  # Set the model to evaluation mode

Preprocess the Input Text

In [ ]:
def preprocess_review(review_text, tokenizer, max_length=512):
    inputs = tokenizer(review_text, padding=True, truncation=True, max_length=max_length, return_tensors="pt")
    inputs = {key: value.to(device) for key, value in inputs.items()}  # Move inputs to GPU if available
    return inputs


In [ ]:
def predict_sentiment(review_text):
    inputs = preprocess_review(review_text, Mytokenizer)

    with torch.no_grad():
        outputs = Mymodel(**inputs)

    logits = outputs.logits
    predictions = torch.argmax(logits, dim=1)  # Get the class with the highest score (0 or 1)

    return "Positive" if predictions.item() == 1 else "Negative"


In [ ]:
# Example review (replace this with the review you want to test)
review_text = "I really enjoyed this movie, it was fantastic!"

# Predict sentiment
sentiment = predict_sentiment(review_text)
print(f"The review sentiment is: {sentiment}")


In [ ]:
# Example review (replace this with the review you want to test)
review_text = " movie achi  thi but  a bit boring and  lenthy"

# Predict sentiment
sentiment = predict_sentiment(review_text)
print(f"The review sentiment is: {sentiment}")


In [ ]:
# Predict the sentiment for the review
review = "I regret purchasing the XYZ phone, as it has been nothing but a disappointment since day one. The phone's performance is sluggish, with frequent lagging and freezing, making simple tasks like texting or browsing the internet a frustrating experience. The battery drains quickly, even with minimal usage, forcing me to carry a charger everywhere I go. Additionally, the camera quality is subpar, producing blurry and washed-out images even in optimal lighting conditions. I've encountered numerous software glitches and bugs, causing the phone to crash unexpectedly. Overall, I'm thoroughly dissatisfied with the XYZ phone's performance and would not recommend it to anyone looking for a reliable and functional device"

# Predict sentiment
sentiment = predict_sentiment(review)
print(f"The review sentiment is: {sentiment}")

In [ ]:
# Predict the sentiment for the review
review = "the is product is good"
# Predict sentiment
sentiment = predict_sentiment(review)
print(f"The review sentiment is: {sentiment}")

In [ ]:
review = "I recently purchased the latest model of XYZ phone, and I couldn't be happier with my decision! The phone's sleek design, stunning display, and lightning-fast performance exceeded all my expectations. The camera quality is phenomenal, capturing every moment in crisp detail. I also love the long-lasting battery life, which keeps me connected throughout the day without needing constant recharging. The user interface is intuitive and easy to navigate, making it a pleasure to use. Overall, this phone has completely revolutionized my mobile experience, and I highly recommend it to anyone in the market for a new device!"
# Predict sentiment
sentiment = predict_sentiment(review)
print(f"The review sentiment is: {sentiment}")

In [ ]:
review = "This LLM is a black box. It generates text, but it's unclear how it arrives at those outputs.  A lack of transparency makes it difficult to trust the information it provides and limits its usefulness for tasks that require reasoning or justification."
# Predict sentiment
sentiment = predict_sentiment(review)
print(f"The review sentiment is: {sentiment}")

In [ ]:
review = "The backpack's main zipper broke within a week of using it lightly for everyday travel. bad product"# Predict sentiment
sentiment = predict_sentiment(review)
print(f"The review sentiment is: {sentiment}")